## Setup

In [352]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys

sys.path.append("..")

### Ezyquant setup

In [354]:
import ezyquant as ez
from src.config import DATABASE_PATH

ez.connect_sqlite(DATABASE_PATH.absolute().as_posix())

## Get Data

In [355]:
import datetime
from ezyquant.reader import _SETDataReaderCached

In [356]:
sdr = _SETDataReaderCached()
start_date_data = datetime.date(2019, 1, 1)  # ISO format: 2019-01-01
start_date_backtest = datetime.date(2020, 1, 1)  # ISO format: 2020-01-01
end_date = datetime.date.fromisoformat(sdr.last_update())

/home/jj/Documents/Files/JJ_work/Chula/CEDT/2025_2/Electives/Algorithmic_Trading/algo-trading-cedt-final-project/.venv/lib/python3.12/site-packages/ezyquant/utils.py:170: UserWarning: Last update is not the same for all tables. Please check your database.
  out = fn(*new_args, **new_kwargs)


In [357]:
ssc = ez.SETSignalCreator(
    start_date=start_date_data.isoformat(),
    end_date=end_date.isoformat(),
    # index_list=["BANK"],
    index_list=["SET50"],
)

In [358]:
open_df = ssc.get_data(field="open", timeframe="daily")
high_df = ssc.get_data(field="high", timeframe="daily")
low_df = ssc.get_data(field="low", timeframe="daily")
close_df = ssc.get_data(field="close", timeframe="daily")

/home/jj/Documents/Files/JJ_work/Chula/CEDT/2025_2/Electives/Algorithmic_Trading/algo-trading-cedt-final-project/.venv/lib/python3.12/site-packages/ezyquant/reader.py:2200: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="backfill").fillna(1)
/home/jj/Documents/Files/JJ_work/Chula/CEDT/2025_2/Electives/Algorithmic_Trading/algo-trading-cedt-final-project/.venv/lib/python3.12/site-packages/ezyquant/reader.py:2200: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="backfill").fillna(1)
/home/jj/Documents/Files/JJ_work/Chula/CEDT/2025_2/Electives/Algorithmic_Trading/algo-trading-cedt-final-project/.venv/lib/python3.12/site-packages/ezyquant/reader.py:2200: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or ob

## Strategy

In [360]:
import vectorbt as vbt

In [361]:
fast_ma = vbt.MA.run(close_df, window=[1, 10, 50, 150], short_name='fast')
slow_ma = vbt.MA.run(close_df, window=[7, 20, 100, 200], short_name='slow')

In [364]:
entries = fast_ma.ma_crossed_above(slow_ma)
# entries = slow_ma.ma_crossed_above(close_df)

In [365]:
exits = fast_ma.ma_crossed_below(slow_ma)
# exits = slow_ma.close_below(close_df)

## Backtesting

In [366]:
pf = vbt.Portfolio.from_signals(
    close_df,
    entries,
    exits,
    freq='1D',
)

In [367]:
print(pf.stats())

Start                                 2019-01-02 00:00:00
End                                   2023-12-28 00:00:00
Period                                 1212 days 00:00:00
Start Value                                         100.0
End Value                                      119.046085
Total Return [%]                                19.046085
Benchmark Return [%]                            15.826896
Max Gross Exposure [%]                          98.958333
Total Fees Paid                                       0.0
Max Drawdown [%]                                 34.60955
Max Drawdown Duration         654 days 00:50:31.578947368
Total Trades                                    38.520833
Total Closed Trades                              38.09375
Total Open Trades                                0.427083
Open Trade PnL                                   1.654929
Win Rate [%]                                    32.432633
Best Trade [%]                                  31.021803
Worst Trade [%

/tmp/ipykernel_263264/1918761186.py:1: UserWarning: Object has multiple columns. Aggregating using <function mean at 0x712dbdcbb6a0>. Pass column to select a single column/group.
  print(pf.stats())


In [372]:
pf.total_return().unstack(level=['fast_window', 'slow_window'])

fast_window,1,10,50,150
slow_window,7,20,100,200
ADVANC,-0.043065,-0.316722,0.260571,0.014019
AOT,-0.239835,-0.206251,-0.075087,-0.017727
AWC,-0.326068,-0.552546,0.019659,-0.158883
BAM,-0.556777,-0.206176,-0.303359,-0.237091
BANPU,0.181547,-0.361186,0.379193,0.384037
...,...,...,...,...
TTB,0.122908,0.402075,-0.110806,-0.166763
TTW,-0.716043,0.034571,-0.291768,0.034392
TU,-0.294002,0.221163,-0.049719,0.106277


In [373]:
a = ((pf.total_return() - pf.total_benchmark_return()) / abs(pf.total_benchmark_return())).unstack(level=['fast_window', 'slow_window'])


heatmap = vbt.plotting.Heatmap(
    data=a.clip(lower=-5, upper=5),
    x_labels=a.columns,
    y_labels=a.index,

)
heatmap.fig

FigureWidget({
    'data': [{'colorscale': [[0.0, '#0d0887'], [0.1111111111111111, '#46039f'],
                             [0.2222222222222222, '#7201a8'], [0.3333333333333333,
                             '#9c179e'], [0.4444444444444444, '#bd3786'],
                             [0.5555555555555556, '#d8576b'], [0.6666666666666666,
                             '#ed7953'], [0.7777777777777778, '#fb9f3a'],
                             [0.8888888888888888, '#fdca26'], [1.0, '#f0f921']],
              'hoverongaps': False,
              'type': 'heatmap',
              'uid': 'b2894529-7d0d-448e-80e7-410fcf8a54fa',
              'x': [(1, 7), (10, 20), (50, 100), (150, 200)],
              'y': array(['ADVANC', 'AOT', 'AWC', 'BAM', 'BANPU', 'BBL', 'BDMS', 'BEAUTY', 'BEM',
                          'BGRIM', 'BH', 'BJC', 'BLA', 'BPP', 'BTS', 'CBG', 'CENTEL', 'COM7',
                          'CPALL', 'CPF', 'CPN', 'CRC', 'DELTA', 'DTAC', 'EA', 'EGCO', 'GLOBAL',
                          'GL

In [376]:
pf[(10, 20, "BGRIM")].plot().show()

In [375]:
import pandas as pd

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(-(pf.total_return() / pf.max_drawdown()) >= 1)

fast_window  slow_window        
1            7            ADVANC    False
                          AOT       False
                          AWC       False
                          BAM       False
                          BANPU     False
                          BBL        True
                          BDMS      False
                          BEAUTY    False
                          BEM       False
                          BGRIM     False
                          BH        False
                          BJC       False
                          BLA       False
                          BPP       False
                          BTS       False
                          CBG        True
                          CENTEL     True
                          COM7       True
                          CPALL     False
                          CPF       False
                          CPN       False
                          CRC       False
                          DELTA      True
 